# 04 — Python Threads, the GIL and Processes

**Target:** Apache Spark 4.2.0 project runtime; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
When do Python threads help, and when do separate processes make more sense?

**Why this matters:** conventional CPython threads overlap I/O well, but the GIL generally limits simultaneous execution of Python bytecode within one process. Native extensions may release it. Spark worker processes are a different layer again.

In [ ]:
import concurrent.futures as cf, time, sys, os, subprocess, tempfile, pathlib
print(sys.version, 'CPU count:', os.cpu_count())
def cpu_work(n=350_000):
    total=0
    for i in range(n): total += (i*i) % 97
    return total
def timed(executor, tasks=4):
    start=time.perf_counter()
    with executor as pool: list(pool.map(cpu_work, [350_000]*tasks))
    return time.perf_counter()-start
print('serial:', timed(cf.ThreadPoolExecutor(max_workers=1)))
print('threads:', timed(cf.ThreadPoolExecutor(max_workers=4)))
# A child script keeps process-based comparison portable on Windows/Jupyter spawn.
child = "import time\ndef work(n):\n total=0\n for i in range(n): total += (i*i) % 97\n return total\nif __name__ == '__main__':\n from concurrent.futures import ProcessPoolExecutor\n start=time.perf_counter()\n with ProcessPoolExecutor(max_workers=4) as pool: list(pool.map(work,[350_000]*4))\n print(time.perf_counter()-start)\n"
with tempfile.TemporaryDirectory() as folder:
    script=pathlib.Path(folder)/'cpu_process_demo.py'
    script.write_text(child,encoding='utf-8')
    result=subprocess.run([sys.executable,str(script)],capture_output=True,text=True,timeout=60)
    print('processes:',result.stdout.strip(),'seconds; exit:',result.returncode)
    if result.returncode:
        raise RuntimeError(f'Process-pool demo failed (exit {result.returncode}):\n{result.stderr}')

## Compare an I/O wait
The next cell uses a short timer to represent waiting. It does not contact external services.

In [ ]:
def wait_a_bit(_): time.sleep(0.2)
for workers in (1,4):
    start=time.perf_counter()
    with cf.ThreadPoolExecutor(max_workers=workers) as pool: list(pool.map(wait_a_bit, range(8)))
    print(f'{workers} threads, 8 waits: {time.perf_counter()-start:.3f}s')

**Expected:** threads overlap waits; CPU-bound pure Python generally does not scale linearly across threads in conventional CPython, while processes can use multiple cores with extra memory/startup cost. Timing is noisy.

**Why:** the GIL is process-local. Spark's parallelism across Python worker processes is not the same thing as a Python thread pool inside one worker. Free-threaded CPython is optional and requires exact compatibility checks.